# Aula 05 — caderno de acompanhamento

**Limpeza e transformação · Programação para Ciência de Dados · 2026.1**

Este caderno acompanha os slides **na mesma numeração**. O rodapé da tela
mostra em que slide a aula está; procure aqui pela seção de mesmo número.

```
 ▸ Slide 18 · A sentinela      ←  o 18 é o número que está no rodapé do slide
```

Nem todo slide tem célula — capa, divisórias e os de conceito puro ficam de
fora, e a numeração pula. Os que **têm** estão marcados com `▸ caderno` no
canto superior direito da tela.

**Rode na ordem.** As células dependem das anteriores: se pular uma, a
seguinte reclama de um nome que não existe. Se você chegou atrasado, use
*Run → Run All Above* e siga daí.

No fim há um **apêndice** com o que não coube em slide — para depois da aula.

<details>
<summary><b>Índice — slide por slide</b> (clique para abrir)</summary>

| slide | assunto | slide | assunto |
|---|---|---|---|
| **4** | Os 5 datasets | **43** | Três formatos |
| **7** | O sintoma · dtypes | **44** | Ato 1 — o palpite |
| **8** | Quem é o culpado? | **45** | Ato 2 — mixed |
| **9** | Contar os culpados | **46** | Ato 3 — o estrago |
| **10** | O describe denuncia | **47** | A receita segura |
| **11** | O raio-x | **48** | O acessador .dt |
| **12** | 27 estados | **49** | Fuso horário |
| **14** | As 4 caras do vazio | **51** | O acessador .str |
| **15** | NaN ≠ NaN | **52** | O espaço invisível |
| **16** | O disfarce | **53** | O acento |
| **17** | na_values | **54** | A função chave |
| **18** | A sentinela | **55** | De 27 para 10 |
| **19** | Onde estão os buracos | **56** | map × replace |
| **21** | dropna | **57** | extract |
| **22** | fillna | **58** | zfill |
| **23** | Média ou mediana | **60** | assign |
| **24** | ffill e interpolate | **61** | where e select |
| **25** | O sensor caiu | **62** | apply é lento |
| **26** | A linha que não existe | **63** | cut × qcut |
| **27** | Imputar por grupo | **64** | pipe |
| **28** | A conta da imputação | **66** | Olhe antes |
| **29** | Guarde a evidência | **67** | IQR |
| **32** | duplicated | **68** | z-score e o cego |
| **33** | O grão da tabela | **69** | Os três tipos |
| **34** | A quase-duplicata | **71** | O preço da decisão |
| **35** | A que não é erro | **73** | loc × iloc |
| **37** | Os tipos | **74** | A cópia |
| **38** | astype × to_numeric | **76** | O pipeline |
| **39** | Dinheiro brasileiro | **77** | A auditoria |
| **40** | category | **78** | Gravar |
| **41** | category com ordem |  |  |

</details>

---
## Preparação

Rode esta antes de a aula começar.

In [1]:
import unicodedata
import numpy as np
import pandas as pd

pd.set_option("display.width", 110)
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

print("pandas", pd.__version__, "· numpy", np.__version__)

pandas 3.0.5 · numpy 2.5.2


---
## ▸ Slide 4 · Os cinco datasets

Os cinco arquivos da aula, ainda brutos.

In [2]:
titanic = pd.read_csv("dados/titanic.csv")
taxis = pd.read_csv("dados/taxis.csv", parse_dates=["pickup", "dropoff"])
sensores = pd.read_csv("dados/sensores.csv", parse_dates=["ts_utc"])

for nome, d in [("vendas-sujas", pd.read_csv("dados/vendas-sujas.csv")),
                ("sensores", sensores), ("rh", pd.read_csv("dados/rh.csv")),
                ("titanic", titanic), ("taxis", taxis)]:
    print(f"{nome:<14} {d.shape[0]:>5} linhas × {d.shape[1]:>2} colunas")

vendas-sujas     627 linhas × 18 colunas
sensores        1081 linhas ×  5 colunas
rh               222 linhas × 10 colunas
titanic          891 linhas × 12 colunas
taxis           6433 linhas × 14 colunas


---
## ▸ Slide 7 · Passo 1 — o sintoma

O `dtypes` diz **que** há um problema. Não diz qual, nem onde.

In [3]:
bruto = pd.read_csv("dados/vendas-sujas.csv")
bruto.dtypes

id_pedido         int64
data_pedido         str
cliente             str
cpf                 str
email               str
telefone            str
cidade              str
uf                  str
categoria           str
produto             str
qtd                 str
valor               str
desconto_pct    float64
frete               str
canal               str
avaliacao       float64
entrega_dias      int64
obs                 str
dtype: object

---
## ▸ Slide 8 · Passo 2 — olhar os valores

Antes de qualquer conversão: **veja o dado com os próprios olhos.**

In [4]:
for col in ["valor", "data_pedido", "qtd", "frete"]:
    print(f"{col:<13}", list(bruto[col].astype("str").unique()[:5]))

valor         ['R$ 659,70', 'R$ 526,52', '3926.19', '332.34', '1286,40']
data_pedido   ['16-07-2026', '02/04/2026', '18-06-2026', '06-06-2026', '2026-05-07']
qtd           ['2', '3', '1', '-', '5']
frete         ['20.14', '6.98', '15.91', '2.44', '12.22']


---
## ▸ Slide 9 · Passo 3 — quantos são

O mesmo comando, dois diagnósticos: exceção a consertar, ou formato a converter.

In [5]:
for col in ["qtd", "frete", "valor"]:
    culpados = bruto[col][pd.to_numeric(bruto[col], errors="coerce").isna() & bruto[col].notna()]
    print(f"\n{col} — {len(culpados)} de {len(bruto)} não viram número:")
    print(culpados.value_counts().head(4).to_string())


qtd — 12 de 627 não viram número:
qtd
-       6
1 pç    4
3 un    2

frete — 17 de 627 não viram número:
frete
       11
n/d     3
-       3

valor — 492 de 627 não viram número:
valor
R$ 370,28      3
R$ 1.357,48    2
R$ 8.258,80    2
R$ 228,68      2


---
## ▸ Slide 10 · Passo 4 — o que o tipo esconde

`avaliacao` é `float64` e parece saudável. **Leia o `min` e o `max`.**

In [6]:
bruto.describe().round(2)

,id_pedido,desconto_pct,avaliacao,entrega_dias
count,627.00,558.00,588.00,627.00
mean,"100,299.35",10.39,-120.88,11.81
std,173.44,9.27,330.89,20.18
min,"100,001.00",0.00,-999.00,1.00
25%,"100,147.50",0.00,2.00,5.00
50%,"100,301.00",10.00,4.00,10.00
75%,"100,448.50",20.00,5.00,15.00
max,"100,600.00",25.00,5.00,365.00


---
## ▸ Slide 11 · O raio-x

A função que vai ser usada o caderno inteiro.

In [7]:
def raio_x(df):
    """Uma linha por coluna: tipo, faltantes, distintos, exemplo e memória."""
    return pd.DataFrame({
        "tipo": df.dtypes.astype(str),
        "faltam": df.isna().sum(),
        "faltam_%": (df.isna().mean() * 100).round(1),
        "distintos": df.nunique(dropna=True),
        "exemplo": [df[c].dropna().iloc[0] if df[c].notna().any() else None
                    for c in df.columns],
        "memória_KB": (df.memory_usage(deep=True)[1:] / 1024).round(1),
    })


raio_x(bruto)

,tipo,faltam,faltam_%,distintos,exemplo,memória_KB
id_pedido,int64,0,0.00,600,100449,4.90
data_pedido,str,4,0.60,344,16-07-2026,36.00
cliente,str,0,0.00,125,Isabela Moraes,43.70
cpf,str,0,0.00,600,737.242.462-15,37.60
email,str,22,3.50,98,isabela.moraes@exemplo.com.br,46.00
telefone,str,27,4.30,563,(54) 92096-9379,37.50
cidade,str,0,0.00,30,Curitiba,38.20
uf,str,0,0.00,27,PR,32.90
categoria,str,0,0.00,21,esporte,38.40
produto,str,0,0.00,22,Bicicleta ergométrica,45.40


---
## ▸ Slide 12 · Vinte e sete estados

A loja vende em dez. Cada grafia diferente é um "estado" novo para o computador.

In [8]:
print("linhas:", len(bruto), "· duplicadas:", bruto.duplicated().sum())
print("valores distintos de uf:", bruto["uf"].nunique())
bruto["uf"].value_counts().head(8)

linhas: 627 · duplicadas: 18
valores distintos de uf: 27


uf
pr           42
RS           36
RS           31
pe           30
rs           29
PR           28
ce           27
São Paulo    26
Name: count, dtype: int64

---
## ▸ Slide 14 · As quatro caras do vazio

`NaN`, `None`, `NaT` e `<NA>` — quatro nomes, uma ideia. O `isna()` vê todos.

In [9]:
vazios = pd.DataFrame({
    "numero": [1.0, np.nan, 3.0],
    "texto": ["a", None, "c"],
    "data": pd.to_datetime(["2026-01-01", None, "2026-03-01"]),
    "inteiro": pd.array([1, None, 3], dtype="Int64"),
})
print(vazios.dtypes.to_dict())
vazios

{'numero': dtype('float64'), 'texto': <StringDtype(storage='python', na_value=nan)>, 'data': dtype('<M8[us]'), 'inteiro': Int64Dtype()}


,numero,texto,data,inteiro
0,1.00,a,2026-01-01,1
1,NaN,NaN,NaT,<NA>
2,3.00,c,2026-03-01,3


In [10]:
vazios.isna()

,numero,texto,data,inteiro
0,False,False,False,False
1,True,True,True,True
2,False,False,False,False


---
## ▸ Slide 15 · NaN não é igual a NaN

O filtro errado **não dá erro** — devolve vazio, e você conclui que está tudo bem.

In [11]:
print("np.nan == np.nan →", np.nan == np.nan)
print("\nfiltrando com == :")
print(bruto[bruto["valor"] == np.nan].shape)
print("\nfiltrando com isna():")
print(bruto[bruto["valor"].isna()].shape)

np.nan == np.nan → False

filtrando com == :
(0, 18)

filtrando com isna():
(3, 18)


---
## ▸ Slide 16 · O disfarce

Dezessete traços derrubam a coluna inteira para texto.

In [12]:
for col in ["email", "telefone", "frete", "qtd", "data_pedido"]:
    disfarce = bruto[col].astype("str").str.strip().isin(["", "-", "n/d", "sem data", "?"])
    print(f"{col:<13} tipo {str(bruto[col].dtype):<6} · {bruto[col].isna().sum():>3} vazios "
          f"reconhecidos · {disfarce.sum():>3} disfarçados")

email         tipo str    ·  22 vazios reconhecidos ·  14 disfarçados
telefone      tipo str    ·  27 vazios reconhecidos ·  13 disfarçados
frete         tipo str    ·  37 vazios reconhecidos ·  17 disfarçados
qtd           tipo str    ·   4 vazios reconhecidos ·   6 disfarçados
data_pedido   tipo str    ·   4 vazios reconhecidos ·   0 disfarçados


---
## ▸ Slide 17 · na_values

O Pandas já reconhece uma lista própria; `na_values` acrescenta a da sua casa.

In [13]:
from pandas.io.parsers.readers import STR_NA_VALUES
print("o que o Pandas já reconhece:", sorted(STR_NA_VALUES))

o que o Pandas já reconhece: ['', '#N/A', '#N/A N/A', '#NA', '-1.#IND', '-1.#QNAN', '-NaN', '-nan', '1.#IND', '1.#QNAN', '<NA>', 'N/A', 'NA', 'NULL', 'NaN', 'None', 'n/a', 'nan', 'null']


In [14]:
NULOS = ["", " ", "  ", "-", "n/d", "sem data", "?", "00/00/0000"]

df = pd.read_csv("dados/vendas-sujas.csv", na_values=NULOS)
print("sem na_values ·", bruto.isna().sum().sum(), "faltantes")
print("com na_values ·", df.isna().sum().sum(), "faltantes")
raio_x(df)[["tipo", "faltam", "faltam_%"]]

sem na_values · 624 faltantes
com na_values · 677 faltantes


,tipo,faltam,faltam_%
id_pedido,int64,0,0.00
data_pedido,str,7,1.10
cliente,str,0,0.00
cpf,str,0,0.00
email,str,36,5.70
telefone,str,40,6.40
cidade,str,0,0.00
uf,str,0,0.00
categoria,str,0,0.00
produto,str,0,0.00


---
## ▸ Slide 18 · A sentinela

Uma nota média de −120,88 numa escala de 1 a 5.

In [15]:
print(df["avaliacao"].value_counts(dropna=False).sort_index())
print("\nmédia com a sentinela:", round(df["avaliacao"].mean(), 2))
print("média sem a sentinela:", round(df["avaliacao"].replace(-999, np.nan).mean(), 2))

avaliacao
-999.00     73
1.00        68
2.00        63
3.00        70
4.00       126
5.00       188
NaN         39
Name: count, dtype: int64

média com a sentinela: -120.88
média sem a sentinela: 3.59


In [16]:
df["avaliacao"] = df["avaliacao"].replace(-999, np.nan)
df["avaliacao"].describe()

count   515.00
mean      3.59
std       1.42
min       1.00
25%       2.00
50%       4.00
75%       5.00
max       5.00
Name: avaliacao, dtype: float64

---
## ▸ Slide 19 · Onde estão os buracos

Os números do mapa de faltantes projetado na tela.

In [17]:
(titanic.isna().sum().to_frame("faltam")
    .assign(**{"%": (titanic.isna().mean() * 100).round(1)})
    .query("faltam > 0"))

,faltam,%
Age,177,19.90
Cabin,687,77.10
Embarked,2,0.20


---
## ▸ Slide 21 · dropna

Sem argumento nenhum, ele apaga 79% da tabela.

In [18]:
print("original           ", titanic.shape)
print("dropna()           ", titanic.dropna().shape)
print("subset=['Embarked']", titanic.dropna(subset=["Embarked"]).shape)
print("how='all'          ", titanic.dropna(how="all").shape)
print("thresh=11          ", titanic.dropna(thresh=11).shape)
print("axis=1             ", titanic.dropna(axis=1).shape)

original            (891, 12)
dropna()            (183, 12)
subset=['Embarked'] (889, 12)
how='all'           (891, 12)
thresh=11           (733, 12)
axis=1              (891, 9)


---
## ▸ Slide 22 · fillna

Uma regra declarada por coluna. Um `fillna(0)` faria cabine virar zero.

In [19]:
amostra = titanic.loc[[5, 17, 19, 26, 28], ["Name", "Age", "Embarked", "Cabin"]]
amostra

,Name,Age,Embarked,Cabin
5,"Moran, Mr. James",NaN,Q,NaN
17,"Williams, Mr. Charles Eugene",NaN,S,NaN
19,"Masselmani, Mrs. Fatima",NaN,C,NaN
26,"Emir, Mr. Farred Chehab",NaN,C,NaN
28,"O'Dwyer, Miss. Ellen ""Nellie""",NaN,Q,NaN


In [20]:
amostra.fillna({"Age": titanic["Age"].median(), "Embarked": "S", "Cabin": "sem registro"})

,Name,Age,Embarked,Cabin
5,"Moran, Mr. James",28.00,Q,sem registro
17,"Williams, Mr. Charles Eugene",28.00,S,sem registro
19,"Masselmani, Mrs. Fatima",28.00,C,sem registro
26,"Emir, Mr. Farred Chehab",28.00,C,sem registro
28,"O'Dwyer, Miss. Ellen ""Nellie""",28.00,Q,sem registro


---
## ▸ Slide 23 · Média ou mediana

O critério é a assimetria. Com `|skew| > 1`, use a mediana.

In [21]:
for nome, s in [("Titanic · Age", titanic["Age"]), ("Titanic · Fare", titanic["Fare"])]:
    print(f"{nome:<16} média {s.mean():>8.2f} · mediana {s.median():>8.2f} · skew {s.skew():>5.2f}")

Titanic · Age    média    29.70 · mediana    28.00 · skew  0.39
Titanic · Fare   média    32.20 · mediana    14.45 · skew  4.79


In [22]:
rh = pd.read_csv("dados/rh.csv", na_values=NULOS)
print(f"salários informados: {rh['salario'].notna().sum()} · em branco: {rh['salario'].isna().sum()}")
print(f"média {rh['salario'].mean():,.2f} · mediana {rh['salario'].median():,.2f} · skew {rh['salario'].skew():.2f}")

pd.DataFrame({
    "original": rh["salario"].describe(),
    "fillna(média)": rh["salario"].fillna(rh["salario"].mean()).describe(),
    "fillna(mediana)": rh["salario"].fillna(rh["salario"].median()).describe(),
})

salários informados: 188 · em branco: 34
média 11,820.80 · mediana 7,869.61 · skew 7.16


,original,fillna(média),fillna(mediana)
count,188.00,222.00,222.00
mean,"11,820.80","11,820.80","11,215.67"
std,"18,807.71","17,300.58","17,359.26"
min,"1,077.40","1,077.40","1,077.40"
25%,"4,934.16","5,461.47","5,461.47"
50%,"7,869.61","9,070.02","7,869.61"
75%,"12,545.58","11,820.80","11,236.79"
max,"182,400.00","182,400.00","182,400.00"


---
## ▸ Slide 24 · ffill, bfill e interpolate

Quatro respostas para o mesmo buraco de uma hora. Repare no `-999` sobrevivendo.

In [23]:
a = sensores[sensores["sensor"] == "A-estufa"].set_index("ts_utc").sort_index()
janela = a.loc["2026-03-02 08:30":"2026-03-02 10:15", "temperatura"]

pd.DataFrame({
    "original": janela,
    "ffill": janela.ffill(),
    "bfill": janela.bfill(),
    "interpolate": janela.interpolate(),
    "média_do_dia": janela.fillna(round(a["temperatura"].mean(), 2)),
})

,original,ffill,bfill,interpolate,média_do_dia
ts_utc,,,,,
2026-03-02 08:30:00+00:00,20.78,20.78,20.78,20.78,20.78
2026-03-02 08:45:00+00:00,21.28,21.28,21.28,21.28,21.28
2026-03-02 09:00:00+00:00,NaN,21.28,21.23,21.27,15.33
2026-03-02 09:15:00+00:00,NaN,21.28,21.23,21.26,15.33
2026-03-02 09:30:00+00:00,NaN,21.28,21.23,21.25,15.33
2026-03-02 09:45:00+00:00,NaN,21.28,21.23,21.24,15.33
2026-03-02 10:00:00+00:00,21.23,21.23,21.23,21.23,21.23
2026-03-02 10:15:00+00:00,-999.00,-999.00,-999.00,-999.00,-999.00


---
## ▸ Slide 25 · O sensor caiu por três horas

O `limit=` é o freio de mão: se o buraco é grande, ele **continua** buraco.

In [24]:
temp = a["temperatura"].replace(-999.0, np.nan)
print("sem limite     · sobra NaN:", temp.ffill().isna().sum())
print("ffill(limit=1) · sobra NaN:", temp.ffill(limit=1).isna().sum())
print("ffill(limit=2) · sobra NaN:", temp.ffill(limit=2).isna().sum())

sem limite     · sobra NaN: 0
ffill(limit=1) · sobra NaN: 3
ffill(limit=2) · sobra NaN: 2


---
## ▸ Slide 26 · A linha que não existe

Se o sensor caiu, não há `NaN` — não há linha. O `asfreq` revela o silêncio.

In [25]:
print(a.index.to_series().diff().value_counts().to_frame("quantas vezes"))
print("\nNaN antes do asfreq: ", temp.isna().sum())
print("NaN depois do asfreq:", temp.asfreq("15min").isna().sum())
temp.asfreq("15min").loc["2026-03-03 11:30":"2026-03-03 15:15"]

                 quantas vezes
ts_utc                        
0 days 00:15:00            348
0 days 00:30:00             11
0 days 03:15:00              1

NaN antes do asfreq:  26
NaN depois do asfreq: 49


ts_utc
2026-03-03 11:30:00+00:00   23.78
2026-03-03 11:45:00+00:00   23.35
2026-03-03 12:00:00+00:00     NaN
2026-03-03 12:15:00+00:00     NaN
2026-03-03 12:30:00+00:00     NaN
2026-03-03 12:45:00+00:00     NaN
2026-03-03 13:00:00+00:00     NaN
2026-03-03 13:15:00+00:00     NaN
2026-03-03 13:30:00+00:00     NaN
2026-03-03 13:45:00+00:00     NaN
2026-03-03 14:00:00+00:00     NaN
2026-03-03 14:15:00+00:00     NaN
2026-03-03 14:30:00+00:00     NaN
2026-03-03 14:45:00+00:00     NaN
2026-03-03 15:00:00+00:00   26.90
2026-03-03 15:15:00+00:00   27.88
Freq: 15min, Name: temperatura, dtype: float64

---
## ▸ Slide 27 · Imputar por grupo

`transform` devolve a mediana do grupo de cada linha, do tamanho da coluna.

In [26]:
titanic.groupby(["Pclass", "Sex"])["Age"].median().unstack()

Sex,female,male
Pclass,,
1,35.00,40.00
2,28.00,30.00
3,21.50,25.00


In [27]:
por_grupo = titanic.groupby(["Pclass", "Sex"])["Age"].transform("median")

comp = titanic.loc[titanic["Age"].isna(), ["Pclass", "Sex"]].copy()
comp["mediana_geral"] = titanic["Age"].median()
comp["mediana_do_grupo"] = por_grupo[titanic["Age"].isna()]
comp.head(8)

,Pclass,Sex,mediana_geral,mediana_do_grupo
5,3,male,28.00,25.00
17,2,male,28.00,30.00
19,3,female,28.00,21.50
26,3,male,28.00,25.00
28,3,female,28.00,21.50
29,3,male,28.00,25.00
31,1,female,28.00,35.00
32,3,female,28.00,21.50


---
## ▸ Slide 28 · A conta da imputação

Toda imputação encolhe a variação. Não é errado — é uma consequência.

In [28]:
titanic["Age_imputada"] = titanic["Age"].fillna(por_grupo)
geral = titanic["Age"].fillna(titanic["Age"].median())

print(f"original     · média {titanic['Age'].mean():.2f} · desvio {titanic['Age'].std():.2f} · faltam {titanic['Age'].isna().sum()}")
print(f"mediana geral· média {geral.mean():.2f} · desvio {geral.std():.2f} · faltam 0")
print(f"por grupo    · média {titanic['Age_imputada'].mean():.2f} · desvio {titanic['Age_imputada'].std():.2f} · faltam 0")

original     · média 29.70 · desvio 14.53 · faltam 177
mediana geral· média 29.36 · desvio 13.02 · faltam 0
por grupo    · média 29.11 · desvio 13.30 · faltam 0


---
## ▸ Slide 29 · Guarde a evidência

Quem não tem idade registrada sobreviveu menos. O buraco carregava informação.

In [29]:
titanic["idade_era_faltante"] = titanic["Age"].isna()
(titanic.groupby("idade_era_faltante")
    .agg(pessoas=("Survived", "size"),
         sobreviveram=("Survived", "mean"),
         tarifa_média=("Fare", "mean"),
         classe_média=("Pclass", "mean")).round(3))

,pessoas,sobreviveram,tarifa_média,classe_média
idade_era_faltante,,,,
False,714,0.41,34.70,2.24
True,177,0.29,22.16,2.60


---
## ▸ Slide 32 · duplicated

`keep=False` é para olhar; `keep="first"` é para remover.

In [30]:
print("keep='first' (padrão) →", df.duplicated().sum())
print("keep='last'           →", df.duplicated(keep="last").sum())
print("keep=False            →", df.duplicated(keep=False).sum())
df[df.duplicated(keep=False)].sort_values("id_pedido")[
    ["id_pedido", "cliente", "cidade", "valor"]].head(6)

keep='first' (padrão) → 18
keep='last'           → 18
keep=False            → 36


,id_pedido,cliente,cidade,valor
491,100029,Sabrina Teixeira,Belo Horizonte,"2810,80"
268,100029,Sabrina Teixeira,Belo Horizonte,"2810,80"
12,100076,Caio Mendonça,Salvador,2672.55
566,100076,Caio Mendonça,Salvador,2672.55
474,100081,Caio Mendonça,Porto Alegre,"R$ 317,99"
471,100081,Caio Mendonça,Porto Alegre,"R$ 317,99"


---
## ▸ Slide 33 · O grão da tabela

609 linhas para 600 pedidos. Sobram nove.

In [31]:
sem_dup = df.drop_duplicates()
print(len(df), "→", len(sem_dup), "linhas")
print("pedidos distintos:", sem_dup["id_pedido"].nunique())

repetidos = sem_dup[sem_dup.duplicated("id_pedido", keep=False)].sort_values("id_pedido")
repetidos[["id_pedido", "cliente", "cidade", "valor"]].head(6)

627 → 609 linhas
pedidos distintos: 600


,id_pedido,cliente,cidade,valor
351,100002,GABRIELA NUNES,SÃO PAULO,"R$ 200,04"
364,100002,gabriela nunes,São Paulo,"R$ 200,04"
203,100084,BRUNO CARVALHO,FORTALEZA,"R$ 3.864,04"
523,100084,Bruno Carvalho,Fortaleza,"R$ 3.864,04"
614,100282,FÁBIO REZENDE,CURITIBA,"R$ 58,14"
103,100282,Fábio Rezende,Curitiba,"R$ 58,14"


---
## ▸ Slide 34 · A quase-duplicata

Comparar por uma **chave normalizada**, não pelo texto bruto.

In [32]:
def chave(s):
    """Texto reduzido ao essencial: sem acento, sem espaço extra, minúsculo."""
    return (s.astype("str")
             .str.normalize("NFKD").str.encode("ascii", "ignore").str.decode("ascii")
             .str.replace(r"\s+", " ", regex=True)
             .str.strip().str.lower())


limpo = sem_dup.assign(_k=chave(sem_dup["cliente"]))
limpo = limpo.drop_duplicates(subset=["id_pedido", "_k", "produto"]).drop(columns="_k")

print(f"627 → {len(sem_dup)} → {len(limpo)} linhas")
print("uma linha por pedido?", limpo["id_pedido"].is_unique)

627 → 609 → 600 linhas
uma linha por pedido? True


---
## ▸ Slide 35 · A duplicata que não é erro

935 "duplicatas" que são corridas distintas — o horário prova.

In [33]:
colunas_valor = ["distance", "fare", "tip", "payment"]
print("olhando só", colunas_valor, "→", taxis.duplicated(colunas_valor).sum())
print("olhando a linha inteira      →", taxis.duplicated().sum())
taxis[taxis.duplicated(colunas_valor, keep=False)].sort_values("fare")[
    ["pickup", "distance", "fare", "tip", "payment"]].head(5)

olhando só ['distance', 'fare', 'tip', 'payment'] → 935
olhando a linha inteira      → 0


,pickup,distance,fare,tip,payment
3377,2019-03-27 15:18:33,0.00,2.50,0.00,cash
120,2019-03-21 17:21:44,0.00,2.50,0.00,cash
4663,2019-03-27 13:57:06,0.00,2.50,0.00,cash
5624,2019-03-11 14:04:50,0.00,2.50,0.00,cash
3889,2019-03-30 06:42:55,0.00,2.50,0.00,cash


---
## ▸ Slide 37 · int64 e Int64

O minúsculo não representa ausência: um buraco e a coluna vira `float64`.

In [34]:
print(pd.Series([1, 2, None]).dtype, "←  int virou float por causa do None")
print(pd.Series([1, 2, None], dtype="Int64").dtype, "  ←  continua inteiro, com <NA>")
pd.Series([1, 2, None], dtype="Int64")

float64 ←  int virou float por causa do None
Int64   ←  continua inteiro, com <NA>


0       1
1       2
2    <NA>
dtype: Int64

---
## ▸ Slide 38 · astype × to_numeric

O rígido falha; o tolerante converte e **você conta** quem ficou pelo caminho.

In [35]:
for descrição, serie in [("com NaN na coluna", limpo["qtd"]),
                         ("sem NaN, com texto", limpo["qtd"].dropna())]:
    try:
        serie.astype("int")
        print(f"{descrição:<20} → converteu")
    except Exception as e:
        print(f"{descrição:<20} → {type(e).__name__}: {e}")

com NaN na coluna    → ValueError: cannot convert float NaN to integer
sem NaN, com texto   → ValueError: invalid literal for int() with base 10: '1 pç'


In [36]:
qtd_num = pd.to_numeric(limpo["qtd"], errors="coerce")
virou_nan = qtd_num.isna() & limpo["qtd"].notna()

print("valores que não viraram número:", virou_nan.sum())
print(limpo.loc[virou_nan, ["id_pedido", "qtd"]].to_string(index=False))

limpo["qtd"] = qtd_num.astype("Int64")

valores que não viraram número: 6
 id_pedido  qtd
    100181 1 pç
    100338 1 pç
    100350 1 pç
    100421 3 un
    100344 1 pç
    100303 3 un


---
## ▸ Slide 39 · Dinheiro brasileiro

A receita ingênua multiplica por cem quem já vinha com ponto decimal.

In [37]:
exemplos = pd.Series(["R$ 1.234,56", "977,76", "1004.84", "R$ -120,00"])
ingenuo = (exemplos.str.replace("R$ ", "", regex=False)
                   .str.replace(".", "", regex=False)
                   .str.replace(",", ".", regex=False))
pd.DataFrame({"bruto": exemplos, "receita ingênua": pd.to_numeric(ingenuo, errors="coerce")})

,bruto,receita ingênua
0,"R$ 1.234,56","1,234.56"
1,"977,76",977.76
2,1004.84,"100,484.00"
3,"R$ -120,00",-120.00


In [38]:
def para_real(s):
    """Converte dinheiro em texto para float, tolerando os dois padrões."""
    t = s.astype("str").str.replace(r"[R$\s]", "", regex=True)
    tem_virgula = t.str.contains(",", na=False)
    t = t.mask(tem_virgula, t.str.replace(".", "", regex=False).str.replace(",", ".", regex=False))
    return pd.to_numeric(t, errors="coerce")


pd.DataFrame({"bruto": exemplos, "receita segura": para_real(exemplos)})

,bruto,receita segura
0,"R$ 1.234,56","1,234.56"
1,"977,76",977.76
2,1004.84,"1,004.84"
3,"R$ -120,00",-120.00


In [39]:
limpo["valor"] = para_real(limpo["valor"])
limpo["valor"].describe()

count       597.00
mean      3,179.40
std      14,003.26
min        -120.00
25%         321.70
50%         988.12
75%       2,795.74
max     209,344.09
Name: valor, dtype: float64

---
## ▸ Slide 40 · category

422,9 MB contra 5,7 MB. Em 600 linhas não muda nada; em seis milhões, muda tudo.

In [40]:
grande = pd.Series(np.random.default_rng(1).choice(limpo["produto"].unique(), 6_000_000))
print(f"6 milhões como str:      {grande.memory_usage(deep=True)/1024**2:8.1f} MB")
print(f"6 milhões como category: {grande.astype('category').memory_usage(deep=True)/1024**2:8.1f} MB")
del grande

6 milhões como str:         423.0 MB
6 milhões como category:      5.7 MB


---
## ▸ Slide 41 · category com ordem

Sem a ordem declarada, o `groupby` devolveria alfabético.

In [41]:
NIVEIS = ["Analista", "Analista Sênior", "Especialista", "Coordenador",
          "Gerente", "Diretor", "Diretora"]
rh["cargo"] = pd.Categorical(rh["cargo"], categories=NIVEIS, ordered=True)

print("acima de Especialista:", int((rh["cargo"] > "Especialista").sum()), "pessoas")
rh.groupby("cargo", observed=True)["salario"].median().round(2)

acima de Especialista: 91 pessoas


cargo
Analista            4,369.72
Analista Sênior     4,902.46
Especialista        8,288.70
Coordenador         9,657.55
Gerente            14,248.76
Diretor           182,400.00
Diretora          164,900.00
Name: salario, dtype: float64

---
## ▸ Slide 43 · Três formatos numa coluna

Um sistema, três exportações, dez anos de história.

In [42]:
limpo["data_pedido"].dropna().sample(8, random_state=3).to_frame()

,data_pedido
358,2026-04-10
414,24/06/2026
38,01-08-2026
354,2026-07-27
349,2026-07-08
385,04/02/2026
135,2026-04-20
274,13/04/2026


---
## ▸ Slide 44 · Ato 1 — o palpite

Ele decide **um** formato e aplica em todas. Perde 534 de 600 — mas avisa.

In [43]:
ingenuo = pd.to_datetime(limpo["data_pedido"], errors="coerce")
print("viraram NaT:", ingenuo.isna().sum(), "de", len(limpo))

viraram NaT: 534 de 600


/var/folders/h6/79_cm06j5nx03vw77nwfsbqr0000gn/T/ipykernel_65549/1123460650.py:1: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  ingenuo = pd.to_datetime(limpo["data_pedido"], errors="coerce")


---
## ▸ Slide 45 · Ato 2 — format="mixed"

Só 12 perdidas. **Olhe o fim do período antes de comemorar.**

In [44]:
misto = pd.to_datetime(limpo["data_pedido"], format="mixed", dayfirst=True, errors="coerce")
print("viraram NaT:", misto.isna().sum(), "de", len(limpo))
print("período:", misto.min().date(), "→", misto.max().date())

viraram NaT: 12 de 600
período: 2026-01-02 → 2026-12-07


---
## ▸ Slide 46 · Ato 3 — o estrago

O `dayfirst=True` valeu também para as datas em ISO: 112 trocaram de mês.

In [45]:
def para_data(s, formatos=("%Y-%m-%d", "%d/%m/%Y", "%d-%m-%Y")):
    """Tenta cada formato explícito, na ordem; só mexe no que ainda está vazio."""
    saida = pd.Series(pd.NaT, index=s.index, dtype="datetime64[us]")
    for f in formatos:
        pendente = saida.isna() & s.notna()
        if not pendente.any():
            break
        saida[pendente] = pd.to_datetime(s[pendente], format=f, errors="coerce")
    return saida


cascata = para_data(limpo["data_pedido"])
divergem = (misto != cascata) & misto.notna() & cascata.notna()

print("NaT na cascata:", cascata.isna().sum())
print("datas em que os dois métodos DISCORDAM:", divergem.sum())
pd.DataFrame({"texto": limpo["data_pedido"], "format='mixed'": misto,
              "cascata explícita": cascata})[divergem].head(8)

NaT na cascata: 12
datas em que os dois métodos DISCORDAM: 112


,texto,format='mixed',cascata explícita
4,2026-05-07,2026-07-05,2026-05-07
12,2026-06-09,2026-09-06,2026-06-09
13,2026-06-01,2026-01-06,2026-06-01
14,2026-02-01,2026-01-02,2026-02-01
18,2026-01-11,2026-11-01,2026-01-11
19,2026-04-02,2026-02-04,2026-04-02
31,2026-07-08,2026-08-07,2026-07-08
32,2026-07-11,2026-11-07,2026-07-11


---
## ▸ Slide 47 · A receita segura

Conferir a faixa pega 30 das 112. As outras 82 ficam ali, plausíveis e erradas.

In [46]:
limpo["data_pedido"] = cascata
ABERTURA, HOJE = pd.Timestamp("2026-01-01"), pd.Timestamp("2026-08-28")

pegas = int((~misto.between(ABERTURA, HOJE) & misto.notna()).sum())
print("cascata      · faixa:", cascata.min().date(), "→", cascata.max().date())
print("format=mixed · fora da faixa do negócio:", pegas, f"de {int(divergem.sum())} corrompidas")
print(f"\no teste de faixa pegaria {pegas}; as outras {int(divergem.sum()) - pegas} passariam batido.")

cascata      · faixa: 2026-01-05 → 2026-08-02
format=mixed · fora da faixa do negócio: 30 de 112 corrompidas

o teste de faixa pegaria 30; as outras 82 passariam batido.


In [47]:
# as 5 datas que não existem no calendário
texto = sem_dup.loc[limpo.index, "data_pedido"]
limpo.loc[cascata.isna() & texto.notna(), ["id_pedido"]].assign(
    texto=texto[cascata.isna() & texto.notna()])

,id_pedido,texto
149,100051,30/02/2026
162,100461,30/02/2026
359,100298,31/02/2026
442,100236,31/02/2026
587,100392,30/02/2026


---
## ▸ Slide 48 · O acessador .dt

Uma data vira dez colunas — e a pergunta "que dia da semana vende mais?".

In [48]:
d = limpo["data_pedido"]
pd.DataFrame({
    "data": d, "ano": d.dt.year, "mês": d.dt.month, "dia": d.dt.day,
    "semana": d.dt.isocalendar().week, "dia_semana": d.dt.dayofweek,
    "nome_do_dia": d.dt.day_name(), "trimestre": d.dt.quarter,
    "fim_de_mês?": d.dt.is_month_end, "dias_no_mês": d.dt.days_in_month,
}).head(6)

,data,ano,mês,dia,semana,dia_semana,nome_do_dia,trimestre,fim_de_mês?,dias_no_mês
0,2026-07-16,"2,026.00",7.00,16.00,29,3.00,Thursday,3.00,False,31.00
1,2026-04-02,"2,026.00",4.00,2.00,14,3.00,Thursday,2.00,False,30.00
2,2026-06-18,"2,026.00",6.00,18.00,25,3.00,Thursday,2.00,False,30.00
3,2026-06-06,"2,026.00",6.00,6.00,23,5.00,Saturday,2.00,False,30.00
4,2026-05-07,"2,026.00",5.00,7.00,19,3.00,Thursday,2.00,False,31.00
5,2026-02-16,"2,026.00",2.00,16.00,8,0.00,Monday,1.00,False,28.00


In [49]:
(limpo.assign(dia=d.dt.day_name())
      .groupby("dia", observed=True)["valor"].agg(pedidos="size", faturamento="sum")
      .sort_values("faturamento", ascending=False).round(2))

,pedidos,faturamento
dia,,
Wednesday,76,"399,130.57"
Friday,67,"352,511.67"
Monday,86,"304,886.41"
Saturday,101,"283,716.10"
Thursday,89,"212,764.35"
Tuesday,95,"170,591.03"
Sunday,74,"154,189.32"


---
## ▸ Slide 49 · Fuso horário

Três horas de erro numa resposta que parecia trivial.

In [50]:
temp = a["temperatura"].replace(-999.0, np.nan).where(lambda s: s < 40)
temp_local = temp.tz_convert("America/Sao_Paulo")

print("hora mais quente lendo o carimbo bruto (UTC):",
      temp.groupby(temp.index.hour).mean().idxmax(), "h")
print("hora mais quente no fuso de Brasília:      ",
      temp_local.groupby(temp_local.index.hour).mean().idxmax(), "h")

hora mais quente lendo o carimbo bruto (UTC): 18 h
hora mais quente no fuso de Brasília:       15 h


In [51]:
# tz_localize ROTULA (não move); tz_convert TRADUZ (move)
sem_fuso = a.index.tz_localize(None)
print("sem fuso nenhum:      ", sem_fuso[0])
print("tz_localize (rotula): ", sem_fuso[:1].tz_localize("America/Sao_Paulo")[0])
print("tz_convert  (traduz): ", a.index[:1].tz_convert("America/Sao_Paulo")[0])

sem fuso nenhum:       2026-03-02 03:00:00
tz_localize (rotula):  2026-03-02 03:00:00-03:00
tz_convert  (traduz):  2026-03-02 00:00:00-03:00


---
## ▸ Slide 51 · O acessador .str

A coluna inteira de uma vez, pulando os `NaN` sozinho.

In [52]:
nomes = limpo["cliente"].head(6)
pd.DataFrame({
    "bruto": nomes,
    "strip": nomes.str.strip(),
    "lower": nomes.str.lower(),
    "title": nomes.str.title(),
    "tamanho": nomes.str.len(),
})

,bruto,strip,lower,title,tamanho
0,Isabela Moraes,Isabela Moraes,isabela moraes,Isabela Moraes,14
1,André Bittencourt,André Bittencourt,andré bittencourt,André Bittencourt,17
2,Rafael Siqueira,Rafael Siqueira,rafael siqueira,Rafael Siqueira,15
3,GABRIELA NUNES,GABRIELA NUNES,gabriela nunes,Gabriela Nunes,14
4,Fábio Rezende,Fábio Rezende,fábio rezende,Fábio Rezende,13
5,Zilda Marques,Zilda Marques,zilda marques,Zilda Marques,13


---
## ▸ Slide 52 · O espaço que ninguém vê

`strip()` só limpa as pontas. O do meio precisa de `\s+`.

In [53]:
sujo = pd.Series(["  São  Paulo ", "Rio\tde Janeiro", "Porto   Alegre"])
pd.DataFrame({
    "bruto": sujo.str.replace(" ", "·"),
    "só_strip": sujo.str.strip().str.replace(" ", "·"),
    "strip_+_regex": sujo.str.replace(r"\s+", " ", regex=True).str.strip().str.replace(" ", "·"),
})

,bruto,só_strip,strip_+_regex
0,··São··Paulo·,São··Paulo,São·Paulo
1,Rio\tde·Janeiro,Rio\tde·Janeiro,Rio·de·Janeiro
2,Porto···Alegre,Porto···Alegre,Porto·Alegre


---
## ▸ Slide 53 · Dois jeitos de escrever "ã"

Visualmente idênticos; byte a byte, diferentes.

In [54]:
composto = "São Paulo"
decomposto = unicodedata.normalize("NFD", composto)

print("iguais?", composto == decomposto, "· tamanhos:", len(composto), len(decomposto))
print("composto:  ", [hex(ord(c)) for c in composto[:3]])
print("decomposto:", [hex(ord(c)) for c in decomposto[:4]])

iguais? False · tamanhos: 9 10
composto:   ['0x53', '0xe3', '0x6f']
decomposto: ['0x53', '0x61', '0x303', '0x6f']


---
## ▸ Slide 54 · A função chave, por dentro

A mesma `chave()` já usada nas duplicatas — agora vendo o efeito passo a passo.

In [55]:
cidades = pd.Series(["São Paulo", "SAO PAULO", " São  Paulo ", "Sao Paulo",
                     "Florianópolis", "Florianopolis", "Goiânia", "GOIANIA"])
pd.DataFrame({"bruto": cidades, "chave": chave(cidades)})

,bruto,chave
0,São Paulo,sao paulo
1,SAO PAULO,sao paulo
2,São Paulo,sao paulo
3,Sao Paulo,sao paulo
4,Florianópolis,florianopolis
5,Florianopolis,florianopolis
6,Goiânia,goiania
7,GOIANIA,goiania


In [56]:
print("valores distintos antes: ", cidades.nunique())
print("valores distintos depois:", chave(cidades).nunique())

valores distintos antes:  8
valores distintos depois: 3


---
## ▸ Slide 55 · De 27 estados para 10

Normalizar resolve a grafia; o **dicionário** resolve o significado.

In [57]:
print("antes:", limpo["uf"].nunique(), "valores distintos")
print(sorted(limpo["uf"].unique()))

antes: 27 valores distintos
[' SP', 'BA', 'CE', 'GO', 'MG', 'Minas Gerais', 'PE', 'PR', 'RJ', 'RS', 'RS ', 'Rio Grande do Sul', 'Rio de Janeiro', 'S.P.', 'SC', 'SP', 'São Paulo', 'ba', 'ce', 'go', 'mg', 'pe', 'pr', 'rj', 'rs', 'sc', 'sp']


In [58]:
MAPA_UF = {"sao paulo": "SP", "s.p.": "SP", "rio de janeiro": "RJ",
           "rio grande do sul": "RS", "minas gerais": "MG"}

limpo["uf"] = chave(limpo["uf"]).replace(MAPA_UF).str.upper()
print("depois:", limpo["uf"].nunique(), "valores distintos")
limpo["uf"].value_counts()

depois: 10 valores distintos


uf
RS    111
SP     93
PR     66
PE     51
CE     50
MG     49
RJ     48
BA     48
SC     44
GO     40
Name: count, dtype: int64

---
## ▸ Slide 56 · map avisa; replace não

A contagem de `NaN` depois do `map` é o teste de que o dicionário está completo.

In [59]:
print("categoria — antes:", limpo["categoria"].nunique())
MAPA_CAT = {"eletronicos": "Eletrônicos", "eletronico": "Eletrônicos",
            "vestuario": "Vestuário", "casa": "Casa", "livros": "Livros",
            "livro": "Livros", "esporte": "Esporte", "esportes": "Esporte"}
limpo["categoria"] = chave(limpo["categoria"]).map(MAPA_CAT)
print("categoria — depois:", limpo["categoria"].nunique(),
      "· sem tradução:", limpo["categoria"].isna().sum())
limpo["categoria"].value_counts()

categoria — antes: 21
categoria — depois: 5 · sem tradução: 0


categoria
Casa           133
Esporte        121
Eletrônicos    117
Livros         116
Vestuário      113
Name: count, dtype: int64

In [60]:
print("departamento no RH — antes:", rh["departamento"].nunique())
MAPA_DEPTO = {
    "ti": "Tecnologia", "t.i.": "Tecnologia", "tecnologia": "Tecnologia",
    "tecnologia da informacao": "Tecnologia",
    "comercial": "Comercial", "vendas": "Comercial",
    "operacoes": "Operações", "financeiro": "Financeiro", "fin.": "Financeiro",
    "pessoas": "Pessoas", "rh": "Pessoas", "r.h.": "Pessoas",
    "recursos humanos": "Pessoas",
}
rh["departamento"] = chave(rh["departamento"]).map(MAPA_DEPTO)
print("departamento — depois:", rh["departamento"].nunique(),
      "· sem tradução:", rh["departamento"].isna().sum())

departamento no RH — antes: 22
departamento — depois: 5 · sem tradução: 0


---
## ▸ Slide 57 · extract com grupos nomeados

Cada grupo da expressão regular vira uma coluna.

In [61]:
cupons = limpo["obs"].str.extract(r"PROMO-(?P<campanha>[A-Z]+)-(?P<codigo>\d{4})")
limpo[["campanha", "cupom"]] = cupons
print(cupons.dropna().head(6).to_string())
limpo["campanha"].value_counts(dropna=False)

   campanha codigo
5     VERAO   6697
8     TROCA   2800
15    TROCA   3399
19    TROCA   7621
29    TROCA   4423
48    TROCA   6856


campanha
NaN      512
TROCA     48
VERAO     13
FRETE     12
BLACK      8
ALUNO      7
Name: count, dtype: int64

---
## ▸ Slide 58 · O CPF que o Excel comeu

Todo CPF tem 11 dígitos. Os de 10 passaram por uma planilha.

In [62]:
cpf = limpo["cpf"].astype("str").str.replace(r"\D", "", regex=True)
print(cpf.str.len().value_counts().sort_index().to_frame("quantos CPFs"))
print("\nos que perderam o zero:")
print(cpf[cpf.str.len() < 11].head(4).to_string())

limpo["cpf"] = cpf.str.zfill(11)
print("\ndepois do zfill:", limpo["cpf"].str.len().value_counts().to_dict())

     quantos CPFs
cpf              
10             11
11            589

os que perderam o zero:
23     2022159085
80     8816458368
181    1365258213
262    9183511350

depois do zfill: {11: 600}


---
## ▸ Slide 60 · assign encadeado

Dentro do `lambda d:`, `d` é a tabela **como está naquele passo**.

In [63]:
limpo["frete"] = limpo["frete"].fillna(0)
limpo["desconto_pct"] = limpo["desconto_pct"].fillna(0)

pedidos = limpo.assign(
    subtotal=lambda d: d["valor"],
    desconto=lambda d: d["valor"] * d["desconto_pct"] / 100,
    liquido=lambda d: d["subtotal"] - d["desconto"] + d["frete"],
    ticket=lambda d: d["liquido"] / d["qtd"],
)
pedidos[["valor", "desconto_pct", "frete", "qtd", "desconto", "liquido", "ticket"]].head()

,valor,desconto_pct,frete,qtd,desconto,liquido,ticket
0,659.70,25.00,20.14,2,164.93,514.92,257.46
1,526.52,0.00,6.98,2,0.00,533.50,266.75
2,"3,926.19",25.00,15.91,3,981.55,"2,960.55",986.85
3,332.34,0.00,2.44,2,0.00,334.78,167.39
4,"1,286.40",10.00,12.22,3,128.64,"1,169.98",389.99


---
## ▸ Slide 61 · np.where e np.select

O `default` é a parte importante: ele entrega o que ninguém previu.

In [64]:
REGIAO = {"SP": "Sudeste", "RJ": "Sudeste", "MG": "Sudeste", "RS": "Sul",
          "SC": "Sul", "PR": "Sul", "BA": "Nordeste", "PE": "Nordeste",
          "CE": "Nordeste", "GO": "Centro-Oeste"}
pedidos["regiao"] = pedidos["uf"].map(REGIAO)
pedidos["porte"] = np.where(pedidos["liquido"] > 1000, "grande", "pequeno")

condicoes = [pedidos["liquido"] >= 50_000, pedidos["liquido"] >= 5_000,
             pedidos["liquido"] >= 500, pedidos["liquido"] >= 0]
pedidos["faixa"] = np.select(condicoes, ["corporativo", "alto", "médio", "baixo"],
                             default="a conferir")
pedidos["faixa"].value_counts()

faixa
médio          320
baixo          211
alto            60
a conferir       5
corporativo      4
Name: count, dtype: int64

In [65]:
# o que o default pegou — e que um default="baixo" teria enterrado
pedidos.loc[pedidos["faixa"] == "a conferir",
            ["id_pedido", "produto", "qtd", "valor", "liquido"]]

,id_pedido,produto,qtd,valor,liquido
30,100367,Colchonete,2,NaN,NaN
106,100310,Calça jeans,2,NaN,NaN
161,100213,Camiseta básica,3,-120.00,-80.31
276,100359,Aspirador vertical,4,-120.00,-102.00
467,100156,Jaqueta corta-vento,1,NaN,NaN


---
## ▸ Slide 62 · apply é lento

A mesma conta, três formas, duas ordens de grandeza.

In [66]:
import time

def cronometra(nome, f, repeticoes=5):
    t = time.perf_counter()
    for _ in range(repeticoes):
        r = f()
    print(f"{nome:<34} {(time.perf_counter() - t) / repeticoes * 1000:7.2f} ms")
    return r


grande = pd.concat([pedidos] * 200, ignore_index=True)
print(f"{len(grande):,} linhas\n")

a1 = cronometra("apply linha a linha (axis=1)", lambda: grande.apply(lambda r: r["valor"] * 1.1, axis=1))
a2 = cronometra("apply na coluna", lambda: grande["valor"].apply(lambda v: v * 1.1))
a3 = cronometra("vetorizado (a coluna inteira)", lambda: grande["valor"] * 1.1)

print("\nos três dão o mesmo resultado:", np.allclose(a1.fillna(0), a3.fillna(0)))
del grande

120,000 linhas

apply linha a linha (axis=1)        196.49 ms
apply na coluna                       8.20 ms
vetorizado (a coluna inteira)         0.05 ms

os três dão o mesmo resultado: True


---
## ▸ Slide 63 · cut × qcut

`cut` corta em valores; `qcut` corta em quantis. Com distribuição torta, muda tudo.

In [67]:
rh["idade"] = ((pd.Timestamp("2026-08-28")
                - pd.to_datetime(rh["nascimento"], format="%d/%m/%Y")).dt.days / 365.25).astype(int)

rh["faixa_etaria"] = pd.cut(rh["idade"], bins=[0, 25, 35, 45, 60, 120],
                            labels=["até 25", "26-35", "36-45", "46-60", "60+"])
rh["faixa_etaria"].value_counts().sort_index().to_frame("pessoas")

,pessoas
faixa_etaria,
até 25,12
26-35,54
36-45,56
46-60,79
60+,21


In [68]:
print("cut(4)  — fatias iguais de VALOR:")
print(pd.cut(rh["salario"], 4).value_counts().sort_index().to_string())
print("\nqcut(4) — fatias iguais de GENTE:")
print(pd.qcut(rh["salario"], 4).value_counts().sort_index().to_string())

cut(4)  — fatias iguais de VALOR:
salario
(896.077, 46408.05]      183
(46408.05, 91738.7]        3
(91738.7, 137069.35]       0
(137069.35, 182400.0]      2

qcut(4) — fatias iguais de GENTE:
salario
(1077.3990000000001, 4934.157]    47
(4934.157, 7869.615]              47
(7869.615, 12545.585]             47
(12545.585, 182400.0]             47


---
## ▸ Slide 64 · pipe

Cada passo com nome. Seis meses depois, a resposta está na lista.

In [69]:
def sem_cancelados(d):
    return d[d["liquido"] > 0]

def com_mes(d):
    return d.assign(mes=d["data_pedido"].dt.to_period("M"))

def so_colunas(d, colunas):
    return d[colunas]


(pedidos
 .pipe(sem_cancelados)
 .pipe(com_mes)
 .pipe(so_colunas, ["mes", "regiao", "liquido"])
 .groupby(["mes", "regiao"], observed=True)["liquido"].sum()
 .unstack().round(0))

regiao,Centro-Oeste,Nordeste,Sudeste,Sul
mes,,,,
2026-01,"14,240.00","45,950.00","232,787.00","44,786.00"
2026-02,"10,586.00","54,487.00","30,247.00","37,015.00"
2026-03,"8,138.00","49,103.00","39,023.00","203,945.00"
2026-04,"2,731.00","44,558.00","40,834.00","50,463.00"
2026-05,"29,043.00","35,968.00","52,886.00","44,600.00"
2026-06,"8,866.00","55,317.00","175,407.00","49,398.00"
2026-07,"15,554.00","30,962.00","52,465.00","77,369.00"
2026-08,NaN,"4,081.00","13,024.00","89,223.00"


---
## ▸ Slide 66 · Olhe antes de decidir

O degrau entre o percentil 99 e o máximo é a assinatura do outlier.

In [70]:
v = pedidos["liquido"]
pd.DataFrame({
    "quantil": [0, .01, .25, .50, .75, .95, .99, 1],
    "valor": [v.quantile(q) for q in [0, .01, .25, .50, .75, .95, .99, 1]],
}).round(2)

,quantil,valor
0,0.00,-102.00
1,0.01,45.12
2,0.25,308.62
3,0.50,900.98
4,0.75,"2,505.76"
5,0.95,"7,369.91"
6,0.99,"18,166.06"
7,1.00,"157,051.98"


---
## ▸ Slide 67 · O método IQR

O corpo dos dados vai do 1º ao 3º quartil; 1,5 amplitudes além, é candidato.

In [71]:
def limites_iqr(s, k=1.5):
    q1, q3 = s.quantile(.25), s.quantile(.75)
    return q1 - k * (q3 - q1), q3 + k * (q3 - q1)


baixo, alto = limites_iqr(v)
fora = (v < baixo) | (v > alto)
print(f"limites IQR: {baixo:,.2f} a {alto:,.2f}")
print(f"fora dos limites: {fora.sum()} pedidos ({fora.mean() * 100:.1f}%)")

limites IQR: -2,987.08 a 5,801.46
fora dos limites: 51 pedidos (8.5%)


---
## ▸ Slide 68 · z-score e o z robusto

O z comum usa média e desvio — as duas coisas que o outlier estraga.

In [72]:
z = (v - v.mean()) / v.std()

def z_robusto(s):
    mediana = s.median()
    mad = (s - mediana).abs().median()
    return 0.6745 * (s - mediana) / mad          # 0,6745 calibra contra a normal


zr = z_robusto(v)
print("z comum   · |z| > 3:", int((z.abs() > 3).sum()), "pedidos")
print("z robusto · |z| > 3:", int((zr.abs() > 3).sum()), "pedidos")

pedidos.assign(z=z.round(2), z_robusto=zr.round(2)).nlargest(5, "liquido")[
    ["id_pedido", "liquido", "z", "z_robusto"]]

z comum   · |z| > 3: 4 pedidos
z robusto · |z| > 3: 83 pedidos


,id_pedido,liquido,z,z_robusto
81,100371,"157,051.98",14.21,150.99
569,100284,"149,086.99",13.47,143.29
392,100057,"115,283.61",10.36,110.60
134,100413,"89,047.46",7.94,85.23
526,100145,"19,093.08",1.50,17.59


---
## ▸ Slide 69 · Os três tipos de outlier

**Leia a coluna `obs`.** É ela que separa o erro do fato.

In [73]:
pedidos.loc[fora, ["id_pedido", "produto", "qtd", "liquido", "obs"]].nlargest(6, "liquido")

,id_pedido,produto,qtd,liquido,obs
81,100371,Bola oficial,70,"157,051.98",venda corporativa - contrato anual
569,100284,Atlas escolar,145,"149,086.99",venda corporativa - contrato anual
392,100057,Bola oficial,167,"115,283.61",venda corporativa - contrato anual
134,100413,Jogo de panelas,119,"89,047.46",venda corporativa - contrato anual
526,100145,Fone Bluetooth,5,"19,093.08",cupom PROMO-FRETE-4363
504,100342,Teclado mecânico,5,"18,373.74",NaN


In [74]:
print("— erro de digitação: qtd 999 —")
print(pedidos.nlargest(3, "qtd")[["id_pedido", "produto", "qtd", "valor"]].to_string(index=False))

print("\n— erro de digitação: prazo de entrega —")
print(pedidos.nlargest(3, "entrega_dias")[["id_pedido", "produto", "entrega_dias"]].to_string(index=False))

print("\n— valor impossível: negativo —")
print(pedidos[pedidos["valor"] < 0][["id_pedido", "produto", "valor"]].to_string(index=False))

— erro de digitação: qtd 999 —
 id_pedido             produto  qtd    valor
    100270 Jaqueta corta-vento  999 1,149.00
    100373   Luminária de mesa  999 1,469.65
    100394     Jogo de panelas  999   350.47

— erro de digitação: prazo de entrega —
 id_pedido           produto  entrega_dias
    100544         Cafeteira           365
    100250      Bola oficial           240
    100386 Python para dados           180

— valor impossível: negativo —
 id_pedido            produto   valor
    100213    Camiseta básica -120.00
    100359 Aspirador vertical -120.00


In [75]:
corporativas = pedidos[pedidos["obs"].str.contains("corporativa", na=False)]
print(f"{len(corporativas)} pedidos corporativos = "
      f"{corporativas['liquido'].sum() / pedidos['liquido'].sum() * 100:.1f}% do faturamento")
corporativas[["id_pedido", "produto", "qtd", "liquido"]]

4 pedidos corporativos = 30.7% do faturamento


,id_pedido,produto,qtd,liquido
81,100371,Bola oficial,70,"157,051.98"
134,100413,Jogo de panelas,119,"89,047.46"
392,100057,Bola oficial,167,"115,283.61"
569,100284,Atlas escolar,145,"149,086.99"


---
## ▸ Slide 71 · O preço da decisão

Quatro respostas defensáveis para "quanto vendemos". Escolha e **escreva por quê**.

In [76]:
def resumo(s, rotulo):
    return pd.Series({"pedidos": s.size, "média": s.mean(), "mediana": s.median(),
                      "total": s.sum(), "desvio": s.std()}, name=rotulo)


pd.DataFrame([
    resumo(v, "tudo"),
    resumo(v[~fora], "removendo pelo IQR"),
    resumo(v.clip(upper=alto), "aparando no limite (clip)"),
    resumo(v[v > 0], "só os positivos"),
]).round(2)

,pedidos,média,mediana,total,desvio
tudo,600.00,"2,784.33",900.98,"1,662,243.26","10,859.69"
removendo pelo IQR,549.00,"1,287.55",745.26,"703,001.43","1,321.64"
aparando no limite (clip),600.00,"1,673.16",900.98,"998,875.82","1,786.58"
só os positivos,595.00,"2,793.99",915.10,"1,662,425.57","10,876.67"


---
## ▸ Slide 73 · loc × iloc

A fatia do `loc` **inclui** o fim; a do `iloc` **exclui**.

In [77]:
amostra = pedidos.head(5)[["id_pedido", "produto", "liquido"]]
amostra.index = ["a", "b", "c", "d", "e"]

print("loc['b':'d'] →", list(amostra.loc["b":"d"].index), " (inclui o fim)")
print("iloc[1:3]    →", list(amostra.iloc[1:3].index), "      (exclui o fim)")
amostra

loc['b':'d'] → ['b', 'c', 'd']  (inclui o fim)
iloc[1:3]    → ['b', 'c']       (exclui o fim)


,id_pedido,produto,liquido
a,100449,Bicicleta ergométrica,514.92
b,100102,Cafeteira,533.50
c,100443,Cafeteira,"2,960.55"
d,100528,Estatística prática,334.78
e,100202,Jaqueta corta-vento,"1,169.98"


---
## ▸ Slide 74 · A cópia e o Copy-on-Write

No pandas 3, escrever numa fatia **nunca** altera o original.

In [78]:
teste = pedidos[["id_pedido", "liquido"]].head(5).copy()
caros = teste[teste["liquido"] > teste["liquido"].median()]
caros["liquido"] = 0                     # sem erro, sem aviso

print("a fatia mudou:")
print(caros.to_string(index=False))
print("\nmas o original NÃO mudou:")
print(teste.to_string(index=False))

a fatia mudou:
 id_pedido  liquido
    100443        0
    100202        0

mas o original NÃO mudou:
 id_pedido  liquido
    100449   514.92
    100102   533.50
    100443 2,960.55
    100528   334.78
    100202 1,169.98


In [79]:
# para alterar de verdade: um comando só, com .loc
teste.loc[teste["liquido"] > teste["liquido"].median(), "liquido"] = 0
print(teste.to_string(index=False))

 id_pedido  liquido
    100449   514.92
    100102   533.50
    100443     0.00
    100528   334.78
    100202     0.00


---
## ▸ Slide 76 · O pipeline inteiro

Tudo o que decidimos, numa função que se pode rodar de novo no mês que vem.

In [80]:
def limpar_vendas(caminho, verbose=True):
    """Do CSV bruto à tabela pronta, com relatório de cada etapa."""
    log = []

    def anota(passo, d):
        log.append((passo, len(d), int(d.isna().sum().sum())))
        return d

    d = anota("1. leitura com na_values", pd.read_csv(caminho, na_values=NULOS))

    d = anota("2. sem linhas idênticas", d.drop_duplicates())
    d = d.assign(_k=chave(d["cliente"]))
    d = anota("3. sem quase-duplicatas",
              d.drop_duplicates(subset=["id_pedido", "_k", "produto"]).drop(columns="_k"))

    d["avaliacao"] = d["avaliacao"].replace(-999, np.nan)
    d["valor"] = para_real(d["valor"])
    d["qtd"] = pd.to_numeric(d["qtd"], errors="coerce").astype("Int64")
    d["data_pedido"] = para_data(d["data_pedido"])
    d = anota("4. tipos convertidos", d)

    d["cliente"] = d["cliente"].str.replace(r"\s+", " ", regex=True).str.strip().str.title()
    d["cidade"] = (d["cidade"].str.replace(r"\s+", " ", regex=True).str.strip().str.title()
                    .str.replace(r"\b(De|Do|Da|Dos|Das)\b", lambda m: m.group().lower(), regex=True))
    d["uf"] = chave(d["uf"]).replace(MAPA_UF).str.upper()
    d["categoria"] = chave(d["categoria"]).map(MAPA_CAT)
    d["canal"] = d["canal"].str.strip().str.lower()
    d["cpf"] = d["cpf"].astype("str").str.replace(r"\D", "", regex=True).str.zfill(11)
    email = d["email"].astype("str").str.strip().str.lower()
    d["email"] = email.where(email.str.fullmatch(r"[\w.+-]+@[\w-]+\.[\w.]+"))
    tel = d["telefone"].astype("str").str.replace(r"\D", "", regex=True).str.replace(r"^55", "", regex=True)
    d["telefone"] = tel.where(tel.str.len().isin([10, 11]))
    d = anota("5. texto padronizado", d)

    d["avaliacao_faltava"] = d["avaliacao"].isna()
    d["frete"] = d["frete"].fillna(0)
    d["desconto_pct"] = d["desconto_pct"].fillna(0)
    d["qtd"] = d["qtd"].fillna(d.groupby("categoria")["qtd"].transform("median")).astype("Int64")
    d = anota("6. faltantes tratados", d)

    d = d.assign(
        liquido=lambda x: x["valor"] * (1 - x["desconto_pct"] / 100) + x["frete"],
        regiao=lambda x: x["uf"].map(REGIAO),
        mes=lambda x: x["data_pedido"].dt.to_period("M"),
        dia_semana=lambda x: x["data_pedido"].dt.day_name(),
        campanha=lambda x: x["obs"].str.extract(r"PROMO-([A-Z]+)-")[0],
        corporativo=lambda x: x["obs"].str.contains("corporativa", na=False),
    )
    d["categoria"] = d["categoria"].astype("category")
    d["uf"] = d["uf"].astype("category")
    d = anota("7. derivadas e categorias", d)

    impossivel = (d["valor"] < 0) | (d["qtd"] > 100) & (~d["corporativo"])
    d.loc[impossivel, ["valor", "liquido"]] = np.nan
    d = anota(f"8. {int(impossivel.sum())} valores impossíveis anulados", d)

    if verbose:
        print(pd.DataFrame(log, columns=["etapa", "linhas", "faltantes"]).to_string(index=False))
    return d


vendas = limpar_vendas("dados/vendas-sujas.csv")

                            etapa  linhas  faltantes
         1. leitura com na_values     627        677
          2. sem linhas idênticas     609        657
          3. sem quase-duplicatas     600        644
             4. tipos convertidos     600        723
             5. texto padronizado     600        723
            6. faltantes tratados     600        590
        7. derivadas e categorias     600       1129
8. 7 valores impossíveis anulados     600       1143


In [81]:
raio_x(vendas)

,tipo,faltam,faltam_%,distintos,exemplo,memória_KB
id_pedido,int64,0,0.00,600,100449,4.70
data_pedido,datetime64[us],12,2.00,190,2026-07-16 00:00:00,4.70
cliente,str,0,0.00,32,Isabela Moraes,41.30
cpf,str,0,0.00,600,73724246215,35.20
email,str,34,5.70,32,isabela.moraes@exemplo.com.br,43.70
telefone,str,40,6.70,560,54920969379,34.10
cidade,str,0,0.00,16,Curitiba,36.60
uf,category,0,0.00,10,PR,1.10
categoria,category,0,0.00,5,Esporte,0.90
produto,str,0,0.00,22,Bicicleta ergométrica,43.40


---
## ▸ Slide 77 · A auditoria

Cada linha é uma promessa. O valor delas aparece no mês que vem.

In [82]:
def auditar(d):
    testes = {
        "uma linha por pedido": d["id_pedido"].is_unique,
        "sem linhas duplicadas": not d.duplicated().any(),
        "data é data": pd.api.types.is_datetime64_any_dtype(d["data_pedido"]),
        "valor é número": pd.api.types.is_numeric_dtype(d["valor"]),
        "UF com 2 letras": d["uf"].dropna().astype("str").str.fullmatch(r"[A-Z]{2}").all(),
        "no máximo 27 UFs": d["uf"].nunique() <= 27,
        "CPF com 11 dígitos": d["cpf"].str.len().eq(11).all(),
        "sem valor negativo": (d["valor"].dropna() >= 0).all(),
        "avaliação entre 1 e 5": d["avaliacao"].dropna().between(1, 5).all(),
        "datas dentro do período": d["data_pedido"].dropna().between(
            pd.Timestamp("2026-01-01"), pd.Timestamp("2026-08-28")).all(),
        "categoria sem buraco": d["categoria"].notna().all(),
        "frete sem buraco": d["frete"].notna().all(),
    }
    return pd.Series(testes).map({True: "✓ passou", False: "✗ FALHOU"}).to_frame("resultado")


auditar(vendas)

,resultado
uma linha por pedido,✓ passou
sem linhas duplicadas,✓ passou
data é data,✓ passou
valor é número,✓ passou
UF com 2 letras,✓ passou
no máximo 27 UFs,✓ passou
CPF com 11 dígitos,✓ passou
sem valor negativo,✓ passou
avaliação entre 1 e 5,✓ passou
datas dentro do período,✓ passou


---
## ▸ Slide 78 · Gravar — e o que se perde

O CSV não guarda tipo. Todo o trabalho de tipagem evapora na gravação.

In [83]:
vendas.to_csv("dados/vendas-limpas.csv", index=False)
relida = pd.read_csv("dados/vendas-limpas.csv")

pd.DataFrame({
    "antes de gravar": vendas.dtypes.astype(str),
    "depois de reler": relida.dtypes.astype(str),
}).loc[["data_pedido", "categoria", "uf", "qtd", "valor"]]

,antes de gravar,depois de reler
data_pedido,datetime64[us],str
categoria,category,str
uf,category,str
qtd,Int64,int64
valor,float64,float64


> Se o destino for outro programa Python, `to_parquet` preserva os tipos,
> ocupa menos e lê mais rápido.

---
---
# Apêndice — o que não coube em slide

Daqui para baixo **não há slide correspondente**. É material para depois da
aula, para a Lista 2 e para consulta.

---
## A.1 — `interpolate` que conhece o relógio

O `interpolate()` puro conta **linhas**; com índice de data, `method="time"`
conta **minutos**. Nas leituras do sensor as linhas não são regulares.

In [84]:
grade = temp.asfreq("15min")
pd.DataFrame({
    "original": grade,
    "linear": grade.interpolate(),
    "por_tempo": grade.interpolate(method="time"),
    "limite_2": grade.interpolate(limit=2),
}).loc["2026-03-03 11:30":"2026-03-03 15:15"].round(2)

,original,linear,por_tempo,limite_2
ts_utc,,,,
2026-03-03 11:30:00+00:00,23.78,23.78,23.78,23.78
2026-03-03 11:45:00+00:00,23.35,23.35,23.35,23.35
2026-03-03 12:00:00+00:00,NaN,23.62,23.62,23.62
2026-03-03 12:15:00+00:00,NaN,23.90,23.90,23.90
2026-03-03 12:30:00+00:00,NaN,24.17,24.17,NaN
2026-03-03 12:45:00+00:00,NaN,24.44,24.44,NaN
2026-03-03 13:00:00+00:00,NaN,24.72,24.72,NaN
2026-03-03 13:15:00+00:00,NaN,24.99,24.99,NaN
2026-03-03 13:30:00+00:00,NaN,25.26,25.26,NaN


---
## A.2 — A árvore de decisão do faltante

| situação | decisão |
|---|---|
| falta em < 5% e a linha não é preciosa | `dropna(subset=[...])` |
| falta em 5–40%, número simétrico | `fillna(média)` |
| falta em 5–40%, número assimétrico | `fillna(mediana)` |
| falta em 5–40% e há grupo natural | `groupby().transform("median")` |
| falta em série temporal ordenada | `ffill(limit=n)` ou `interpolate` |
| falta em > 50% | descarte a coluna, ou vire uma flag |
| falta porque **não se aplica** | preencha com a categoria "não se aplica" |

A última linha é a mais esquecida: `data_cancelamento` vazia num pedido
entregue não é buraco nenhum — é a resposta certa.

---
## A.3 — `convert_dtypes`: útil como sugestão, nunca como decisão

In [85]:
vendas.convert_dtypes().dtypes.to_frame("sugestão do Pandas").head(12)

,sugestão do Pandas
id_pedido,Int64
data_pedido,datetime64[us]
cliente,string
cpf,string
email,string
telefone,string
cidade,string
uf,category
categoria,category
produto,string


Ele acerta o óbvio e não sabe que `avaliacao` é uma escala de 1 a 5, nem que
`uf` tem 27 valores possíveis. **O tipo certo depende do significado**, e
significado não se infere de um arquivo.

---
## A.4 — `contains` e o `NaN`: o que mudou no pandas 3

No tipo `str` novo, `contains` já devolve `False` para o vazio. No velho
`object` — que você vai encontrar em código de antes — devolve `NaN`.

In [86]:
como_str = vendas["obs"].str.contains("PROMO")
como_object = vendas["obs"].astype(object).str.contains("PROMO")

print("dtype str    →", como_str.dtype, "· tem NaN?", como_str.isna().any())
print("dtype object →", como_object.dtype, "· tem NaN?", como_object.isna().any())
print("\npor isso o hábito de escrever na=False continua valendo.")

dtype str    → bool · tem NaN? False
dtype object → object · tem NaN? True

por isso o hábito de escrever na=False continua valendo.


---
## A.5 — Validar com expressão regular: e-mail e telefone

In [87]:
email = vendas["email"].astype("str").str.strip().str.lower()
valido = email.str.fullmatch(r"[\w.+-]+@[\w-]+\.[\w.]+")
print("e-mails válidos:", int(valido.sum()), "· inválidos ou vazios:", int(len(email) - valido.sum()))

tel = vendas["telefone"].astype("str").str.replace(r"\D", "", regex=True)
print("\ncomprimento dos telefones:", tel.str.len().value_counts().to_dict())

e-mails válidos: 566 · inválidos ou vazios: 34

comprimento dos telefones: {11.0: 560}


---
## A.6 — Titanic: a tabela pronta em oito linhas

Tudo o que a aula ensinou, aplicado de uma vez.

In [88]:
t = (pd.read_csv("dados/titanic.csv")
       .drop(columns=["Cabin", "Ticket"])
       .dropna(subset=["Embarked"])
       .assign(
           idade_faltava=lambda d: d["Age"].isna(),
           Age=lambda d: d["Age"].fillna(d.groupby(["Pclass", "Sex"])["Age"].transform("median")),
           faixa=lambda d: pd.cut(d["Age"], [0, 12, 18, 35, 60, 100],
                                  labels=["criança", "adolescente", "adulto jovem",
                                          "adulto", "idoso"]),
           sozinho=lambda d: (d["SibSp"] + d["Parch"]) == 0,
       ))
print(t.shape, "· faltantes restantes:", t.isna().sum().sum())
t.groupby(["faixa", "Sex"], observed=True)["Survived"].agg(pessoas="size", taxa="mean").round(3)

(889, 13) · faltantes restantes: 0


pessoas  taxa
faixa        Sex                  
criança      female       32  0.59
             male         37  0.57
adolescente  female       36  0.75
             male         34  0.09
adulto jovem female      173  0.75
             male        341  0.16
adulto       female       69  0.77
             male        146  0.20
idoso        female        2  1.00
             male         19  0.10

---
## A.7 — Táxis: seis perguntas que o `isna()` não responde

Valores **presentes e coerentes com o tipo**, e incoerentes com o mundo.

In [89]:
tx = taxis.assign(
    minutos=lambda d: (d["dropoff"] - d["pickup"]).dt.total_seconds() / 60,
    payment=lambda d: d["payment"].fillna("não informado"),
)
pd.Series({
    "distância zero": (tx["distance"] == 0).sum(),
    "duração zero ou negativa": (tx["minutos"] <= 0).sum(),
    "zero passageiros": (tx["passengers"] == 0).sum(),
    "mais de uma hora": (tx["minutos"] > 60).sum(),
    "gorjeta maior que a corrida": (tx["tip"] > tx["fare"]).sum(),
    "total ≠ tarifa+gorjeta+pedágio": (
        (tx["total"] - (tx["fare"] + tx["tip"] + tx["tolls"])).abs() > 0.01).sum(),
}).to_frame("corridas")

,corridas
distância zero,51
duração zero ou negativa,6
zero passageiros,96
mais de uma hora,50
gorjeta maior que a corrida,0
total ≠ tarifa+gorjeta+pedágio,6411


Uma delas dá **zero**, e isso também é resposta. A última é a interessante: a
conta não fecha em 6.411 das 6.433 corridas. Antes de gritar "erro", olhe o
**tamanho** da diferença:

In [90]:
(tx["total"] - (tx["fare"] + tx["tip"] + tx["tolls"])).round(2).value_counts().head(6).to_frame("corridas")

,corridas
3.30,2625
3.80,1510
4.30,838
0.80,629
1.30,333
1.80,236


**A diferença assume meia dúzia de valores fixos.** Não é dado corrompido: são
as taxas de Nova York — sobretaxa do MTA, taxa de congestionamento, adicional
noturno. A coluna que explicaria a conta não veio no arquivo.

*Um analista apressado "corrigiria" 6.411 linhas. O certo era perguntar.*

---
## A.8 — A coluna derivada que denuncia

Nenhum campo do arquivo é a velocidade. Ela nasce de dois outros.

In [91]:
velocidade = (tx["distance"] / (tx["minutos"] / 60)).replace([np.inf, -np.inf], np.nan)
print("mediana:", round(velocidade.median(), 1), "milhas por hora — trânsito de Manhattan")
print("acima de 100 mph:", int((velocidade > 100).sum()), "corridas")
tx.assign(vel=velocidade.round(1)).nlargest(5, "vel")[
    ["pickup", "distance", "minutos", "fare", "vel"]]

mediana:

 10.0 milhas por hora — trânsito de Manhattan
acima de 100 mph: 11 corridas


,pickup,distance,minutos,fare,vel
770,2019-03-02 03:16:59,9.40,0.12,2.50,"4,834.30"
6089,2019-03-17 23:16:34,10.10,0.30,12.00,"2,020.00"
491,2019-03-07 07:11:33,1.60,0.10,2.50,960.00
5284,2019-03-22 06:24:02,1.70,0.15,2.50,680.00
3241,2019-03-24 17:19:27,0.34,0.03,65.00,612.00


Nove milhas em sete segundos. O taxímetro registrou a distância certa e o
relógio errado. A **coluna derivada denunciou o que as duas originais
escondiam** — e essa é a razão de derivar antes de confiar.

---
# As dez regras

1. **Descreva antes de mexer.** `dtypes`, `unique`, `describe`, `isna().sum()`,
   `duplicated().sum()`.
2. **O vazio tem disfarces:** `""`, `"-"`, `-999`, e a linha que nem existe.
3. **Imputar é afirmar algo sobre o mundo** — e encolhe a variância. Guarde a
   flag `_faltava`.
4. **Duplicata só existe depois que você define o grão** da tabela.
5. **Converta com `errors="coerce"` e conte** o que virou `NaN`. Nunca em silêncio.
6. **Data: formato explícito, `dayfirst` consciente, e confira a faixa.**
7. **Normalize antes de comparar:** acento, caixa e espaço.
8. **Vetorizado primeiro;** `apply(axis=1)` é o último recurso.
9. **Outlier não se remove por fórmula.** Erro de digitação, evento raro e
   valor impossível pedem coisas diferentes.
10. **Escreva o pipeline como função e audite o resultado.** Limpeza que não se
    repete não é limpeza, é conserto.

---

### Próximo passo

`exercicio-aula05.ipynb` — a Lista 2, com um dataset que você ainda não viu.